# 🎥 Video 02: Molecular File Formats and I/O Operations
**RDKit Masterclass — From Zero to Drug Discovery AI**
* **Duration:** ~25 - 30 Minutes
* **Curriculum Level:** LEVEL 1 (Topic 06: Working with Molecular Files) & LEVEL 5 (Topic 25)
* **Target Audience:** Cheminformatics Engineers, Computational Chemists, Data Scientists

---

### ⏱️ Video Agenda & Timestamps
* **00:00 - 04:30** | Chemical File Formats Explained: SDF, MOL (V2000 vs V3000), SMI, CSV
* **04:30 - 09:30** | Reading Single Structures: `MolFromMolFile` and `MolFromMolBlock`
* **09:30 - 15:00** | High-Throughput Reading: `SDMolSupplier` vs `ForwardSDMolSupplier`
* **15:00 - 19:30** | Reading and Writing SD Tags (Metadata & Properties)
* **19:30 - 24:00** | Writing Chemical Files: `SDWriter` & `SmilesWriter`
* **24:00 - 27:30** | Pandas Integration with `PandasTools`
* **27:30 - 30:00** | Hands-On Challenge: Processing and Filtering a Multi-Compound SDF

---

### 🎯 Learning Objectives
1. Understand the anatomy of MDL MOL and SD (Structure-Data) files.
2. Read chemical files safely with `SDMolSupplier`, handling `None` molecules without halting execution.
3. Stream massive multi-gigabyte SDF files using generator-based `ForwardSDMolSupplier`.
4. Access and write property metadata tags (e.g., biological activity, compound IDs, assay results).
5. Load and save molecular datasets directly to/from Pandas DataFrames with `PandasTools`.

In [1]:
# Step 0: Imports and Environment Setup
import os
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, PandasTools
from rdkit.Chem.Draw import IPythonConsole
import pandas as pd

IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = (320, 220)

print(f"RDKit Version: {rdkit.__version__}")

RDKit Version: 2026.03.6


## 1. Anatomy of a MOL Block & V2000 Format
An MDL MOL block contains:
1. **Header Block (3 lines):** Molecule name, user/software info, comments.
2. **Counts Line:** Number of atoms, bonds, chiral flag, etc.
3. **Atom Block:** X, Y, Z coordinates and atom symbols.
4. **Bond Block:** Connected atom indices (1-indexed) and bond types.
5. **Properties/M END Line:** Indicates end of the block.

In [2]:
# Generate and print a MOL block for Paracetamol
paracetamol = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")
AllChem.Compute2DCoords(paracetamol)
mol_block = Chem.MolToMolBlock(paracetamol)

print("=== MDL MOL BLOCK (V2000) ===")
print("\n".join(mol_block.splitlines()[:18]))  # Print header, counts, and initial atom lines

=== MDL MOL BLOCK (V2000) ===

     RDKit          2D

 11 11  0  0  0  0  0  0  0  0999 V2000
   -4.2313   -0.2780    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.7824    0.1103    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.3942    1.5592    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
   -1.7218   -0.9504    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0
   -0.2729   -0.5622    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.1153    0.8867    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.5642    1.2750    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.6249    0.2143    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    4.0738    0.6025    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    2.2367   -1.2346    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.7878   -1.6228    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
  1  2  1  0
  2  3  2  0
  2  4  1  0


## 2. Reading Multi-Molecule SDF Files with `SDMolSupplier`
An **SDF (Structure-Data File)** bundles multiple MOL blocks separated by `$$$$`, with each molecule having associated key-value metadata tags (`> <TAG_NAME>`).

In [3]:
# Create a sample SDF file in memory for practice
sample_sdf_path = "sample_drugs.sdf"

sdf_content = """
  Aspirin
     RDKit          2D

  9  9  0  0  0  0  0  0  0  0999 V2000
    0.0000    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.2990    0.7500    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.2990    2.2500    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    2.5981    0.0000    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    3.8971    0.7500    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.1962    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.1962   -1.5000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    3.8971   -2.2500    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.5981   -1.5000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
  1  2  1  0
  2  3  2  0
  2  4  1  0
  4  5  1  0
  5  6  2  0
  6  7  1  0
  7  8  2  0
  8  9  1  0
  9  4  1  0
M  END
> <COMPOUND_ID>
DRUG_001

> <TARGET_IC50_nM>
25.4

> <MECHANISM>
COX-1/COX-2 Inhibitor

$$$$
"""

with open(sample_sdf_path, "w") as f:
    f.write(sdf_content)

# Reading with SDMolSupplier
suppl = Chem.SDMolSupplier(sample_sdf_path)
print(f"Total entries in supplier: {len(suppl)}")

mol = suppl[0]
if mol:
    print(f"Loaded molecule with {mol.GetNumAtoms()} atoms.")
    print("Metadata properties:", mol.GetPropsAsDict())

Total entries in supplier: 1


[19:26:11] ERROR: Counts line too short: '' on line4
[19:26:11] ERROR: moving to the beginning of the next molecule


## 3. High-Throughput / Big Data: `ForwardSDMolSupplier`
`SDMolSupplier` indexes the entire file into memory (which fails for multi-gigabyte files like the entire ZINC or PubChem databases).
`ForwardSDMolSupplier` is a streaming generator that reads one molecule at a time from a file stream, using constant minimal memory!

In [4]:
# Memory-efficient forward supplier demonstration
with open(sample_sdf_path, "rb") as f_stream:
    forward_suppl = Chem.ForwardSDMolSupplier(f_stream)
    for idx, mol in enumerate(forward_suppl):
        if mol is None:
            continue
        props = mol.GetPropsAsDict()
        print(f"Streamed Molecule {idx+1}: ID={props.get('COMPOUND_ID', 'N/A')}, IC50={props.get('TARGET_IC50_nM', 'N/A')} nM")

[19:26:11] ERROR: Counts line too short: '' on line4
[19:26:11] ERROR: moving to the beginning of the next molecule


## 4. Writing Molecules and Properties with `SDWriter`
Let's prepare multiple molecules with custom properties and export them to a new production-ready `.sdf` file.

In [5]:
compounds_to_export = [
    ("Erlotinib", "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1", 12.0, "Approved"),
    ("Gefitinib", "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN4CCOCC4", 3.2, "Approved"),
    ("Lapatinib", "CS(=O)(=O)CCNCc1ccc(o1)c2ccc3c(c2)c(c(cn3)c4cccc(c4Cl)Oc5cccc(c5)F)N", 9.8, "Approved"),
    ("Afatinib", "CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(Cl)c(F)c3)ncnc2cc1O[C@H]4CCOC4", 0.5, "Approved")
]

output_sdf = "kinase_inhibitors.sdf"
writer = Chem.SDWriter(output_sdf)

for name, smi, ic50, status in compounds_to_export:
    m = Chem.MolFromSmiles(smi)
    if m:
        # Generate 2D coordinates for clear visual depiction in chemical software (ChemDraw, PyMOL, Discovery Studio)
        AllChem.Compute2DCoords(m)
        
        # Set metadata properties
        m.SetProp("_Name", name)
        m.SetProp("Compound_Name", name)
        m.SetProp("SMILES", smi)
        m.SetProp("EGFR_IC50_nM", str(ic50))
        m.SetProp("Clinical_Status", status)
        
        writer.write(m)

writer.close()
print(f"✅ Successfully wrote {len(compounds_to_export)} molecules to '{output_sdf}'!")

✅ Successfully wrote 4 molecules to 'kinase_inhibitors.sdf'!


## 5. Working with SMILES Files (`SmilesMolSupplier` and `SmilesWriter`)
SMILES files (`.smi` or `.tsv`) are tab/space-separated text files containing:
`SMILES <whitespace> Molecule_Name`.

In [6]:
smi_file = "compounds.smi"
with open(smi_file, "w") as f:
    f.write("CC(=O)Oc1ccccc1C(=O)O Aspirin\n")
    f.write("CC(C)Cc1ccc(cc1)C(C)C(=O)O Ibuprofen\n")
    f.write("CC(=O)Nc1ccc(O)cc1 Paracetamol\n")

# Reading SMILES files
smi_suppl = Chem.SmilesMolSupplier(smi_file, delimiter=" ", smilesColumn=0, nameColumn=1, titleLine=False)
for mol in smi_suppl:
    if mol:
        print(f"Read from .smi: {mol.GetProp('_Name'):<12} -> Formula: {Chem.rdMolDescriptors.CalcMolFormula(mol)}")

Read from .smi: Aspirin      -> Formula: C9H8O4
Read from .smi: Ibuprofen    -> Formula: C13H18O2
Read from .smi: Paracetamol  -> Formula: C8H9NO2


## 6. Integration with Pandas: `PandasTools`
`rdkit.Chem.PandasTools` allows you to load SDFs into Pandas DataFrames, rendering interactive 2D structure images directly inside Jupyter tables!

In [7]:
# Load our generated SDF into a Pandas DataFrame
df_sdf = PandasTools.LoadSDF(output_sdf)
print(f"DataFrame Shape: {df_sdf.shape}")
df_sdf[["Compound_Name", "EGFR_IC50_nM", "Clinical_Status", "ROMol"]].head()

DataFrame Shape: (4, 6)


,Compound_Name,EGFR_IC50_nM,Clinical_Status,ROMol
0,Erlotinib,12.0,Approved,<rdkit.Chem.rdchem.Mol object at 0x71c2af884120>
1,Gefitinib,3.2,Approved,<rdkit.Chem.rdchem.Mol object at 0x71c2af884a50>
2,Lapatinib,9.8,Approved,<rdkit.Chem.rdchem.Mol object at 0x71c2af884ac0>
3,Afatinib,0.5,Approved,<rdkit.Chem.rdchem.Mol object at 0x71c2af884ba0>


## 7. 🎯 Video Hands-On Challenge & Solution
### Problem:
1. Load `kinase_inhibitors.sdf` using `SDMolSupplier`.
2. Compute the Molecular Weight (MW) for each compound.
3. Add a new tag `Potency_Tier`:
   * `"High"` if `EGFR_IC50_nM < 5.0`
   * `"Moderate"` otherwise.
4. Save the annotated molecules into `kinase_inhibitors_annotated.sdf`.
5. Verify the new file by reading and printing the tags.

In [8]:
from rdkit.Chem import Descriptors

suppl = Chem.SDMolSupplier(output_sdf)
out_annotated = "kinase_inhibitors_annotated.sdf"
writer = Chem.SDWriter(out_annotated)

for mol in suppl:
    if mol is None:
        continue
    
    # Calculate MW
    mw = Descriptors.MolWt(mol)
    mol.SetDoubleProp("MolecularWeight", round(mw, 2))
    
    # Check potency
    ic50 = float(mol.GetProp("EGFR_IC50_nM"))
    tier = "High" if ic50 < 5.0 else "Moderate"
    mol.SetProp("Potency_Tier", tier)
    
    writer.write(mol)

writer.close()

# Verify output
verify_suppl = Chem.SDMolSupplier(out_annotated)
print("=== VERIFICATION OF ANNOTATED SDF ===")
for m in verify_suppl:
    if m:
        props = m.GetPropsAsDict()
        name = m.GetProp("_Name") if m.HasProp("_Name") else props.get("Compound_Name", "Unknown")
        print(f"Compound: {name:<12} | MW: {props.get('MolecularWeight', 'N/A')} | IC50: {props.get('EGFR_IC50_nM', 'N/A')} nM | Tier: {props.get('Potency_Tier', 'N/A')}")

=== VERIFICATION OF ANNOTATED SDF ===
Compound: Erlotinib    | MW: 393.44 | IC50: 12.0 nM | Tier: Moderate
Compound: Gefitinib    | MW: 446.91 | IC50: 3.2 nM | Tier: High
Compound: Lapatinib    | MW: 566.05 | IC50: 9.8 nM | Tier: Moderate
Compound: Afatinib     | MW: 485.95 | IC50: 0.5 nM | Tier: High


## 8. Summary & Best Practices
* **Use `SDMolSupplier`** when you need fast random access and the file fits in RAM.
* **Use `ForwardSDMolSupplier`** when streaming massive library downloads (millions of compounds).
* **Always call `AllChem.Compute2DCoords(mol)`** before writing to SDF so external cheminformatics tools display clean structures.
* **Use `PandasTools`** for rapid exploratory data analysis, filtering, and interactive visualization.

**Next Up in Video 03:** **Molecular Sanitization, Cleaning & Quality Control** — fixing valence errors, stripping counterion salts, and building an automated dataset curation pipeline!